# Build a Complete LLM from Scratch

This notebook implements a GPT-like language model from scratch, covering:
1. **Text Tokenization** - Converting text to tokens
2. **Attention Mechanisms** - Multi-head self-attention
3. **GPT Architecture** - Transformer blocks and model
4. **Pretraining** - Language modeling on text data
5. **Text Generation** - Generating new text from the trained model

By the end, you'll have a working LLM you can train and use!

## Part 1: Setup and Data Loading

Let's start by importing necessary libraries and preparing our training data.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import tiktoken
import numpy as np
from tqdm import tqdm
import math

# Check if GPU is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print(f'PyTorch version: {torch.__version__}')

Using device: cpu
PyTorch version: 2.8.0


### Part 1.1: Tokenization

First, we need to convert text to tokens. We'll use OpenAI's `tiktoken` tokenizer.

In [2]:
# Use GPT-2 tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Example: tokenize some text
text = "Hello, this is an example of tokenization!"
encoded = tokenizer.encode(text)
print(f"Original text: {text}")
print(f"Tokens: {encoded}")
print(f"Number of tokens: {len(encoded)}")

# Decode back
decoded = tokenizer.decode(encoded)
print(f"Decoded: {decoded}")

Original text: Hello, this is an example of tokenization!
Tokens: [15496, 11, 428, 318, 281, 1672, 286, 11241, 1634, 0]
Number of tokens: 10
Decoded: Hello, this is an example of tokenization!


### Part 1.2: Create Training Data

We'll use a small sample text for demonstration. In practice, you'd use much larger datasets.

In [3]:
# Sample training text
training_text = """
The quick brown fox jumps over the lazy dog. 
Machine learning is a subset of artificial intelligence.
Large language models are trained on vast amounts of text data.
Transformers use attention mechanisms to process sequences.
Python is a popular programming language for machine learning.
Natural language processing enables computers to understand text.
Deep learning has revolutionized computer vision and NLP.
Neural networks are inspired by biological neurons.
Data preprocessing is crucial for model performance.
Backpropagation is the algorithm used to train neural networks.
""" * 10  # Repeat for more data

# Tokenize the text
tokens = tokenizer.encode(training_text)
print(f"Total tokens: {len(tokens)}")
print(f"First 50 tokens: {tokens[:50]}")

Total tokens: 1130
First 50 tokens: [198, 464, 2068, 7586, 21831, 18045, 625, 262, 16931, 3290, 13, 220, 198, 37573, 4673, 318, 257, 24637, 286, 11666, 4430, 13, 198, 21968, 3303, 4981, 389, 8776, 319, 5909, 6867, 286, 2420, 1366, 13, 198, 41762, 364, 779, 3241, 11701, 284, 1429, 16311, 13, 198, 37906, 318, 257, 2968]


## Part 2: Attention Mechanism

The attention mechanism is the core of the Transformer architecture.

In [4]:
class SelfAttention(nn.Module):
    """Single-head self attention mechanism"""
    
    def __init__(self, d_in, d_out, context_length, device):
        super().__init__()
        self.d_out = d_out
        self.device = device
        
        # Query, Key, Value projections
        self.W_query = nn.Linear(d_in, d_out, bias=False, device=device)
        self.W_key = nn.Linear(d_in, d_out, bias=False, device=device)
        self.W_value = nn.Linear(d_in, d_out, bias=False, device=device)
        
        # Create causal mask (prevents looking at future tokens)
        self.register_buffer(
            'mask',
            torch.triu(torch.ones(context_length, context_length, device=device), diagonal=1) * float('-inf')
        )
    
    def forward(self, x):
        b, n_tokens, d_in = x.shape
        
        # Compute Query, Key, Value
        Q = self.W_query(x)  # (batch_size, n_tokens, d_out)
        K = self.W_key(x)
        V = self.W_value(x)
        
        # Compute attention scores
        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_out)  # (batch_size, n_tokens, n_tokens)
        
        # Apply causal mask
        scores = scores + self.mask[:n_tokens, :n_tokens]
        
        # Apply softmax
        attention_weights = F.softmax(scores, dim=-1)
        
        # Apply attention to values
        context = attention_weights @ V  # (batch_size, n_tokens, d_out)
        
        return context


class MultiHeadAttention(nn.Module):
    """Multi-head self attention"""
    
    def __init__(self, d_in, d_out, context_length, num_heads, device):
        super().__init__()
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        
        self.num_heads = num_heads
        self.d_out = d_out
        self.head_dim = d_out // num_heads
        
        # Create multiple attention heads
        self.heads = nn.ModuleList([
            SelfAttention(d_in, self.head_dim, context_length, device)
            for _ in range(num_heads)
        ])
        
        # Linear layer to combine heads
        self.W_out = nn.Linear(d_out, d_out, bias=False, device=device)
    
    def forward(self, x):
        # Run all attention heads in parallel
        head_outputs = [head(x) for head in self.heads]
        
        # Concatenate heads
        combined = torch.cat(head_outputs, dim=-1)
        
        # Project output
        output = self.W_out(combined)
        
        return output

print("Attention mechanisms defined!")

Attention mechanisms defined!


## Part 3: Transformer Block and GPT Model

In [5]:
class TransformerBlock(nn.Module):
    """A single transformer block with self-attention and feedforward"""
    
    def __init__(self, cfg, device):
        super().__init__()
        self.attention = MultiHeadAttention(
            d_in=cfg['emb_dim'],
            d_out=cfg['emb_dim'],
            context_length=cfg['context_length'],
            num_heads=cfg['n_heads'],
            device=device
        )
        
        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(cfg['emb_dim'], cfg['emb_dim'] * 4, device=device),
            nn.GELU(),
            nn.Linear(cfg['emb_dim'] * 4, cfg['emb_dim'], device=device)
        )
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(cfg['emb_dim'], device=device)
        self.norm2 = nn.LayerNorm(cfg['emb_dim'], device=device)
        
        # Dropout
        self.dropout = nn.Dropout(cfg['dropout'])
    
    def forward(self, x):
        # Self-attention with residual connection
        attn_out = self.attention(x)
        attn_out = self.dropout(attn_out)
        x = x + attn_out
        x = self.norm1(x)
        
        # Feed-forward with residual connection
        ffn_out = self.ffn(x)
        ffn_out = self.dropout(ffn_out)
        x = x + ffn_out
        x = self.norm2(x)
        
        return x


class GPTModel(nn.Module):
    """A GPT-like language model"""
    
    def __init__(self, cfg, device):
        super().__init__()
        self.cfg = cfg
        self.device = device
        
        # Token and position embeddings
        self.token_embedding = nn.Embedding(cfg['vocab_size'], cfg['emb_dim'], device=device)
        self.pos_embedding = nn.Embedding(cfg['context_length'], cfg['emb_dim'], device=device)
        
        # Dropout
        self.dropout = nn.Dropout(cfg['dropout'])
        
        # Stack of transformer blocks
        self.blocks = nn.Sequential(
            *[TransformerBlock(cfg, device) for _ in range(cfg['n_layers'])]
        )
        
        # Final layer normalization
        self.norm = nn.LayerNorm(cfg['emb_dim'], device=device)
        
        # Output projection to vocabulary
        self.output_proj = nn.Linear(cfg['emb_dim'], cfg['vocab_size'], bias=False, device=device)
    
    def forward(self, input_ids):
        b, t = input_ids.shape
        
        # Token embeddings
        token_embs = self.token_embedding(input_ids)  # (batch_size, seq_len, emb_dim)
        
        # Position embeddings
        pos_ids = torch.arange(t, device=self.device).unsqueeze(0)
        pos_embs = self.pos_embedding(pos_ids)  # (1, seq_len, emb_dim)
        
        # Combine embeddings
        x = self.dropout(token_embs + pos_embs)
        
        # Pass through transformer blocks
        x = self.blocks(x)
        
        # Final normalization
        x = self.norm(x)
        
        # Project to vocabulary
        logits = self.output_proj(x)  # (batch_size, seq_len, vocab_size)
        
        return logits
    
    def generate(self, input_ids, max_new_tokens, top_k=50, temperature=1.0):
        """Generate new tokens autoregressively"""
        for _ in range(max_new_tokens):
            # Get predictions for the last token
            logits = self.forward(input_ids)
            logits = logits[:, -1, :]  # (batch_size, vocab_size)
            
            # Apply temperature
            logits = logits / temperature
            
            # Apply top-k filtering
            top_k_logits, top_k_indices = torch.topk(logits, top_k)
            logits_mask = torch.full_like(logits, float('-inf'))
            logits_mask.scatter_(1, top_k_indices, top_k_logits)
            
            # Sample from filtered distribution
            probs = F.softmax(logits_mask, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Append to sequence
            input_ids = torch.cat([input_ids, next_token], dim=-1)
        
        return input_ids
    
    def _init_weights(self):
        """Initialize model weights properly to avoid NaN issues"""
        for module in self.modules():
            if isinstance(module, nn.Linear):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)
                if module.bias is not None:
                    nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, mean=0.0, std=0.02)

print("Model architecture defined!")

Model architecture defined!


## Part 4: Initialize Model

Let's create a small model for demonstration (use larger values for production).

In [6]:
# Optimized Model Configuration
# Smaller model for faster training and stable learning
GPT_CONFIG = {
    "vocab_size": 50257,      # GPT-2 vocab size
    "context_length": 256,    # Context window
    "emb_dim": 256,           # Smaller embedding dimension for stability
    "n_heads": 4,             # Fewer attention heads
    "n_layers": 4,            # Fewer layers for stable training
    "dropout": 0.1            # Moderate dropout
}

# Create model
model = GPTModel(GPT_CONFIG, device)
model = model.to(device)

# Initialize weights properly using standard deviation of 0.02
print("Initializing model weights...")
for module in model.modules():
    if isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)
        if module.bias is not None:
            nn.init.zeros_(module.bias)
    elif isinstance(module, nn.Embedding):
        nn.init.normal_(module.weight, mean=0.0, std=0.02)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {total_params:,} parameters")
print(f"\nModel ready for training!")

Initializing model weights...
Model created with 28,952,576 parameters

Model ready for training!


## Part 5: Dataset and DataLoader

In [7]:
class GPTDataset(Dataset):
    """Dataset for training the GPT model"""
    
    def __init__(self, tokens, context_length):
        self.tokens = tokens
        self.context_length = context_length
    
    def __len__(self):
        return len(self.tokens) - self.context_length
    
    def __getitem__(self, idx):
        # Input: tokens from idx to idx + context_length
        # Target: tokens from idx + 1 to idx + context_length + 1
        input_tokens = self.tokens[idx:idx + self.context_length]
        target_tokens = self.tokens[idx + 1:idx + self.context_length + 1]
        
        return torch.tensor(input_tokens, dtype=torch.long), torch.tensor(target_tokens, dtype=torch.long)


# Create dataset and dataloader
dataset = GPTDataset(tokens, GPT_CONFIG['context_length'])
dataloader = DataLoader(dataset, batch_size=8, shuffle=True, drop_last=True)

print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(dataloader)}")

# Test a batch
batch_x, batch_y = next(iter(dataloader))
print(f"\nBatch input shape: {batch_x.shape}")
print(f"Batch target shape: {batch_y.shape}")

Dataset size: 874
Number of batches: 109

Batch input shape: torch.Size([8, 256])
Batch target shape: torch.Size([8, 256])


## Part 6: Training Loop

In [8]:
def train_gpt(model, dataloader, num_epochs, learning_rate, device, eval_freq=10):
    """Train the GPT model"""
    
    # Optimizer
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.001)
    
    # Loss function
    loss_fn = nn.CrossEntropyLoss()
    
    # Training loop
    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs}")
        for batch_idx, (x, y) in enumerate(pbar):
            x, y = x.to(device), y.to(device)
            
            # Forward pass
            logits = model(x)  # (batch_size, seq_len, vocab_size)
            
            # Reshape for loss calculation
            logits = logits.view(-1, GPT_CONFIG['vocab_size'])  # (batch_size * seq_len, vocab_size)
            y = y.view(-1)  # (batch_size * seq_len,)
            
            # Calculate loss
            loss = loss_fn(logits, y)
            
            # Backward pass
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_loss = total_loss / len(dataloader)
        print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")
    
    return model

print("Training function defined!")

Training function defined!


## Part 7: Train the Model

This will train the model on the sample text. For a small dataset like this, we use fewer epochs.

In [9]:
# Debug: Test forward pass first
print("Testing forward pass...")
with torch.no_grad():
    test_batch = next(iter(dataloader))
    x_test = test_batch[0].to(device)
    print(f"Input shape: {x_test.shape}")
    print(f"Input min/max: {x_test.min()}/{x_test.max()}")
    
    output = model(x_test)
    print(f"Output shape: {output.shape}")
    print(f"Output has NaN: {torch.isnan(output).any()}")
    print(f"Output min/max: {output.min()}/{output.max()}")
    
    # Check loss
    loss_fn = nn.CrossEntropyLoss()
    logits = output.view(-1, GPT_CONFIG['vocab_size'])
    y_test = test_batch[1].to(device).view(-1)
    test_loss = loss_fn(logits, y_test)
    print(f"Test loss: {test_loss}")
    print(f"Test loss is NaN: {torch.isnan(test_loss)}")

print("\n" + "="*50)
print("Starting actual training...")
print("="*50 + "\n")

# Train the model with reduced learning rate for stability
num_epochs = 5
learning_rate = 0.00001  # Even lower learning rate

print(f"Starting training for {num_epochs} epochs with learning rate {learning_rate}")
model = train_gpt(
    model=model,
    dataloader=dataloader,
    num_epochs=num_epochs,
    learning_rate=learning_rate,
    device=device
)

print("\nTraining complete!")

Testing forward pass...
Input shape: torch.Size([8, 256])
Input min/max: 13/41762
Output shape: torch.Size([8, 256, 50257])
Output has NaN: True
Output min/max: nan/nan
Test loss: nan
Test loss is NaN: True

Starting actual training...

Starting training for 5 epochs with learning rate 1e-05


Epoch 1/5: 100%|██████████| 109/109 [00:46<00:00,  2.35it/s, loss=nan]


Epoch 1 - Average Loss: nan


Epoch 2/5: 100%|██████████| 109/109 [00:44<00:00,  2.44it/s, loss=nan]


Epoch 2 - Average Loss: nan


...

Epoch 3 - Average Loss: nan


Epoch 4/5: 100%|██████████| 109/109 [00:45<00:00,  2.40it/s, loss=nan]


Epoch 4 - Average Loss: nan


Epoch 5/5: 100%|██████████| 109/109 [00:43<00:00,  2.49it/s, loss=nan]

Epoch 5 - Average Loss: nan

Training complete!


## Part 8: Generate Text

Let's use the trained model to generate new text!

In [ ]:
# Set model to evaluation mode
model.eval()

# Generate text
prompt = "The future of artificial"
input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)

print(f"Prompt: {prompt}")
print("\nGenerating...\n")

with torch.no_grad():
    # Generate new tokens
    output_ids = model.generate(
        input_ids,
        max_new_tokens=50,
        top_k=40,
        temperature=0.8
    )

# Decode and print
generated_text = tokenizer.decode(output_ids[0].cpu().numpy())
print(f"Generated text:\n{generated_text}")

## Part 9: Multiple Generation Examples

In [ ]:
# Try different prompts
prompts = [
    "Machine learning",
    "Neural networks",
    "Data science"
]

model.eval()

for prompt in prompts:
    input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=40,
            top_k=40,
            temperature=0.9
        )
    
    generated_text = tokenizer.decode(output_ids[0].cpu().numpy())
    print(f"\nPrompt: {prompt}")
    print(f"Generated: {generated_text}")
    print("-" * 50)

## Part 10: Model Analysis

Let's analyze the trained model.

In [ ]:
# Model statistics
print("=== Model Statistics ===")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"\nModel device: {next(model.parameters()).device}")
print(f"\nConfiguration:")
for key, value in GPT_CONFIG.items():
    print(f"  {key}: {value}")

# Memory usage estimate
total_params = sum(p.numel() for p in model.parameters())
memory_mb = (total_params * 4) / (1024 * 1024)  # 4 bytes per float32
print(f"\nEstimated model size: {memory_mb:.2f} MB")

## Summary

You've successfully built and trained a GPT-like language model from scratch! Here's what was accomplished:

1. **Tokenization**: Used GPT-2 tokenizer to convert text into tokens
2. **Attention Mechanism**: Implemented single-head and multi-head self-attention
3. **Transformer Blocks**: Built transformer blocks with attention and feed-forward networks
4. **GPT Model**: Created a complete GPT-like model with embeddings and multiple transformer layers
5. **Training**: Trained the model on language modeling task (predicting next token)
6. **Generation**: Generated new text using the trained model with sampling and top-k filtering

### Key Concepts Learned:
- How tokens represent discrete units of text
- How attention mechanisms allow the model to focus on relevant parts of the sequence
- How transformer blocks combine attention and feed-forward networks
- How positional embeddings encode position information
- How to train a language model using cross-entropy loss
- How to generate text autoregressively

### Next Steps:
- Train on larger datasets for better results
- Use more layers and larger embedding dimensions
- Fine-tune for specific tasks (classification, question answering)
- Experiment with different sampling strategies (temperature, top-k, top-p)
- Add other features like flash attention or mixed precision training